# Explore beam transport with ImpactX

See [A Beam Transport Line](https://blast-warpx.github.io/warpx-tutorials/a-beam-transport-line.html) for setup and interpretation, or the [LLNL workshop](https://blast-warpx.github.io/warpx-tutorials/llnl-hpc-2026.html#exercise-3-track-an-electron-beam-through-a-real-beamline) for workshop instructions. This notebook runs the simulation and plots its diagnostics.


In [ ]:
import json
import sys
import tempfile
import time
from contextlib import chdir
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import openpmd_api as io
from impactx import ImpactX, distribution, elements, twiss
from scipy.constants import c, e, m_e

# Locate this notebook's folder even if the kernel started in its parent.
notebook_dir = Path.cwd()
if not (notebook_dir / "impactx_helpers.py").is_file():
    notebook_dir = next(
        (
            notebook_dir / name
            for name in ("beam-transport", "beamline_impactx")
            if (notebook_dir / name / "impactx_helpers.py").is_file()
        ),
        notebook_dir,
    )
sys.path.insert(0, str(notebook_dir))

# Download the upstream lattice once; keep any existing local copy.
lattice_file = notebook_dir / "htu_lattice.py"
if not lattice_file.exists():
    url = "https://raw.githubusercontent.com/BLAST-ImpactX/impactx/26.09/examples/htu_beamline/htu_lattice.py"
    with urlopen(url, timeout=30) as response:
        lattice_file.write_bytes(response.read())

from htu_lattice import get_lattice  # isort: skip
from impactx_helpers import beam_explorer, plot_beam_sizes  # isort: skip

In [ ]:
particles = 10_000
# Bunch and reference total energy, including rest energy.
reference_total_energy_MeV = 100.0

## 1 · Build and run the ImpactX simulation

Set the particle count and total energy above. Run the definition cell, then call the function below.


In [ ]:
def run_beam(particles=10_000, reference_total_energy_MeV=100.0, cpu_threads=2):
    """Run ImpactX in this kernel and save the result in a fresh folder."""
    mass_MeV = m_e * c**2 / e / 1e6
    if particles < 2:
        raise ValueError("Use at least two particles.")
    if (
        not np.isfinite(reference_total_energy_MeV)
        or reference_total_energy_MeV <= mass_MeV
    ):
        raise ValueError(
            "Reference total energy must be finite and exceed the electron rest energy."
        )
    run_root = (
        notebook_dir if notebook_dir.name == "beam-transport" else notebook_dir.parent
    )
    runs = run_root / "runs"
    runs.mkdir(exist_ok=True)
    run = Path(tempfile.mkdtemp(prefix="transport-", dir=runs))
    performance = []

    case = run / f"{reference_total_energy_MeV:g}MeV"
    case.mkdir()
    started = time.perf_counter()
    # ImpactX writes diagnostics relative to its working directory.
    # chdir restores the notebook directory even if the simulation fails.
    with chdir(case):
        sim = ImpactX()
        try:
            # 1. Configure independent-particle tracking and diagnostics.
            sim.omp_threads = cpu_threads  # OpenMP threads, read by init_grids()
            sim.particle_shape = 2
            sim.space_charge = False
            sim.slice_step_diagnostics = True
            sim.init_grids()

            # 2. Set the electron reference particle (kinetic energy in MeV).
            ref = sim.beam.ref
            ref.set_charge_qe(-1.0).set_mass_MeV(mass_MeV).set_kin_energy_MeV(
                reference_total_energy_MeV - mass_MeV
            )

            # 3. Sample a 25 pC Gaussian bunch using Twiss parameters.
            bg = np.sqrt((reference_total_energy_MeV / mass_MeV) ** 2 - 1)
            sigma_t, sigma_pt = 1e-6, 0.025
            bunch = distribution.Gaussian(
                **twiss(
                    beta_x=0.002,
                    beta_y=0.002,
                    beta_t=sigma_t / sigma_pt,
                    emitt_x=1.5e-6 / bg,
                    emitt_y=1.5e-6 / bg,
                    emitt_t=sigma_t * sigma_pt,
                )
            )
            sim.add_particles(bunch_charge=25e-12, distr=bunch, npart=particles)

            # 4. Load the magnets and add entrance/exit snapshots.
            monitor = elements.BeamMonitor("monitor", backend="h5")
            sim.lattice.extend(
                [
                    monitor,
                    *get_lattice("impactx"),
                    monitor,
                ]
            )

            # 5. Track through the lattice and close diagnostic files.
            sim.track_particles()
        finally:
            sim.finalize()

    elapsed = time.perf_counter() - started
    size = sum(p.stat().st_size for p in case.rglob("*") if p.is_file()) / 2**20
    performance.append(
        dict(
            particles=particles,
            total_energy_MeV=reference_total_energy_MeV,
            requested_cpu_threads=cpu_threads,
            elapsed_seconds=elapsed,
            output_MiB=size,
        )
    )
    print(
        f"{reference_total_energy_MeV:g} MeV | {particles:,} particles | {elapsed:.2f} s | {size:.1f} MiB"
    )

    (run / "performance.json").write_text(json.dumps(performance, indent=2) + "\n")
    return run

### Run and measure

Each call saves diagnostics and `performance.json` in a fresh folder under `../runs/`.


In [ ]:
run = run_beam(particles, reference_total_energy_MeV)

## 2 · Open a diagnostic and plot the beam

Load the `beam` species at `TCPhosphor` into a table, with one row per macroparticle.


In [ ]:
energy = reference_total_energy_MeV
screen_name = "TCPhosphor"
case = run / f"{energy:g}MeV"
screen_file = case / "diags/openPMD" / f"{screen_name}.h5"
series = io.Series(str(screen_file), io.Access.read_only)
try:
    iteration = min(series.iterations)
    beam = series.iterations[iteration].particles["beam"].to_df()
finally:
    series.close()
beam[["position_x", "position_y", "weighting"]].head()

Plot transverse positions in millimeters, weighted by charge magnitude in pC per bin. Change the screen name to inspect another location.


In [ ]:
x = beam["position_x"].to_numpy()
y = beam["position_y"].to_numpy()
w = beam["weighting"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
hist = ax.hist2d(x * 1e3, y * 1e3, bins=50, weights=w * e * 1e12, cmap="magma")
fig.colorbar(hist[3], ax=ax, label="Charge per bin (pC)")
ax.set(xlabel="x (mm)", ylabel="y (mm)", title=f"{screen_name} · {energy:g} MeV")
ax.set_aspect("equal")
plt.show()

## 3 · Explore the beam screens

Use **Play screens** or the slider. Turn off **Zoom to fit** to compare sizes on fixed axes. The viewer also saves `beam_explorer.html` in the run folder.


In [ ]:
fig = plot_beam_sizes(run)
beam_explorer(run)